# DieCasting 데이터 확인 및 전처리

1. 라이브러리 세팅  
2. 데이터 로드 및 기본 확인  
3. 결측치 확인
4. 결측치 중앙값(median) 대체
5. 불량 이진화 (0=양품, 1/2/3=불량 → 1)
6. 이상치 탐색

## 1) 라이브러리 세팅

In [1]:
# 라이브러리
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)

DATA_PATH = r"DieCasting_Quality_Raw_Data.csv"

## 2) 데이터 로드

In [2]:
# 데이터 로드 (2줄 헤더 처리)
df_raw = pd.read_csv(DATA_PATH, header=[0, 1])

# MultiIndex 컬럼 평탄화
df_raw.columns = [f"{a}_{b}" for a, b in df_raw.columns]

# 공백 제거
df_raw.columns = [c.replace(" ", "") for c in df_raw.columns]

df_raw.head()

,Process_id,Process_Product_Type,Process_Shot,Process_Velocity_1,Process_Velocity_2,Process_Velocity_3,Process_High_Velocity,Process_Cylinder_Pressure,Process_Rapid_Rise_Time,Process_Biscuit_Thickness,Process_Clamping_Force,Process_Cycle_Time,Process_Pressure_Rise_Time,Process_Casting_Pressure,Process_Spray_Time,Process_Spray_1_Time,Process_Spray_2_Time,Sensor_Melting_Furnace_Temp,Sensor_Air_Pressure,Sensor_Air_Pressure_Min,Sensor_Air_Pressure_Max,Sensor_Coolant_Temp,Sensor_Coolant_Temp_Min,Sensor_Coolant_Temp_Max,Sensor_Coolant_Pressure,Sensor_Factory_Temp,Sensor_Factory_Temp_Min,Sensor_Factory_Temp_Max,Sensor_Factory_Humidity,Sensor_Factory_Humidity_Min,Sensor_Factory_Humidity_Max,Defects_Short_Shot_1,Defects_Bubble_1,Defects_Exfoliation_1,Defects_Blow_Hole_1,Defects_Stain_1,Defects_Dent_1,Defects_Deformation_1,Defects_Contamination_1,Defects_Impurity_1,Defects_Crack_1,Defects_Scratch_1,Defects_Buring_Mark_1,Defects_Inclusions_1,Defects_Short_Shot_2,Defects_Bubble_2,Defects_Exfoliation_2,Defects_Blow_Hole_2,Defects_Stain_2,Defects_Dent_2,Defects_Deformation_2,Defects_Contamination_2,Defects_Impurity_2,Defects_Crack_2,Defects_Scratch_2,Defects_Buring_Mark_2,Defects_Inclusions_2
0,1,1,1,0.144,0.170,0.188,2.134,214,0.008,10,258,20.7,0.044,1037,7.8,0.7,0.8,695.0,6.3,3,9,26.0,10,50,2.71,32.9,18.0,22.0,58.4,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1002,1,2,0.144,0.170,0.182,2.124,217,0.008,11,257,20.7,0.044,1052,7.8,0.7,0.8,696.4,6.3,3,9,26.1,10,50,2.69,32.9,18.0,22.0,58.2,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2003,1,3,0.144,0.170,0.182,2.116,214,0.008,11,257,20.8,0.041,1037,7.8,0.7,0.8,696.4,6.3,3,9,26.1,10,50,2.69,32.9,18.0,22.0,58.2,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3004,1,4,0.144,0.170,0.182,2.137,217,0.008,11,257,20.7,0.043,1051,7.8,0.7,0.8,696.4,6.3,3,9,26.1,10,50,2.69,32.9,18.0,22.0,58.2,18.0,22.0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4005,1,5,0.144,0.172,0.176,2.111,217,0.008,12,257,20.7,0.042,1052,7.8,0.7,0.8,697.9,6.4,3,9,26.1,10,50,2.69,32.9,18.0,22.0,57.8,18.0,22.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 3) 데이터 기본 확인

In [3]:
print("Shape:", df_raw.shape)
print("결측치 상위 20개 컬럼:")
display(df_raw.isna().sum().sort_values(ascending=False).head(20))


Shape: (7535, 57)
결측치 상위 20개 컬럼:


Sensor_Factory_Temp            90
Sensor_Factory_Humidity        90
Sensor_Factory_Temp_Max        90
Sensor_Factory_Humidity_Max    90
Sensor_Factory_Humidity_Min    90
Sensor_Factory_Temp_Min        90
Process_Shot                    0
Process_Product_Type            0
Process_id                      0
Process_Biscuit_Thickness       0
Process_Clamping_Force          0
Process_Cycle_Time              0
Process_Pressure_Rise_Time      0
Process_Casting_Pressure        0
Process_Spray_Time              0
Process_Spray_1_Time            0
Process_Spray_2_Time            0
Sensor_Melting_Furnace_Temp     0
Sensor_Air_Pressure             0
Process_Velocity_1              0
dtype: int64

## 4) 결측치 중앙값(median) 대체

In [4]:
from sklearn.impute import SimpleImputer

df = df_raw.copy()

# 수치형 컬럼만 선택
num_cols = df.select_dtypes(include=np.number).columns

imputer = SimpleImputer(strategy="median")

df[num_cols] = imputer.fit_transform(df[num_cols])

print("수치형 결측치 총합 (전처리 후):", df[num_cols].isna().sum().sum())

수치형 결측치 총합 (전처리 후): 0


## 5) 불량 이진 타겟 생성 (0=양품, 1/2/3=불량)

In [5]:
# Defects 컬럼만 추출
defect_cols = [c for c in df.columns if c.startswith("Defects_")]

# 0이면 양품, 1/2/3 등 0보다 크면 불량(1)
df["defect_any"] = ((df[defect_cols].sum(axis=1)) > 0).astype(int)

# 확인
total = len(df)
n_bad = df["defect_any"].sum()
n_good = total - n_bad

print(f"전체 Shot : {total}")
print(f"불량 Shot : {n_bad} ({n_bad/total*100:.2f}%)")
print(f"양품 Shot : {n_good} ({n_good/total*100:.2f}%)")

전체 Shot : 7535
불량 Shot : 1689 (22.42%)
양품 Shot : 5846 (77.58%)


## 6) 이상치 탐색

모든 수치형 컬럼에 대해 IQR 기준 이상치 개수 + 비율(%)

In [16]:
import pandas as pd
import numpy as np

# 수치형 컬럼 중에서 불량(Defects_*) + 타겟(defect_any) + is_outlier제외
numeric_cols = [
    col for col in df.select_dtypes(include=[np.number]).columns
    if (not col.startswith('Defects'))
    and (col != 'defect_any')
]

results = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_mask = (df[col] < lower) | (df[col] > upper)
    outlier_count = int(outlier_mask.sum())
    outlier_ratio = (outlier_count / len(df)) * 100

    results.append({
        'column': col,
        'outlier_count': outlier_count,
        'outlier_ratio(%)': round(outlier_ratio, 2),
        'lower_bound': round(lower, 4),
        'upper_bound': round(upper, 4)
    })

outlier_df = (
    pd.DataFrame(results)
    .sort_values(by='outlier_ratio(%)', ascending=False)
    .reset_index(drop=True)
)

outlier_df.head(10)

,column,outlier_count,outlier_ratio(%),lower_bound,upper_bound
0,is_outlier,1689,22.42,0.0000,0.0000
1,Sensor_Factory_Temp,374,4.96,29.2000,36.4000
2,Process_Velocity_2,184,2.44,0.1600,0.1760
3,Process_Biscuit_Thickness,12,0.16,2.0000,26.0000
4,Process_Cycle_Time,11,0.15,-1.6000,58.4000
5,Process_High_Velocity,10,0.13,1.5505,3.1065
6,Process_Rapid_Rise_Time,10,0.13,0.0020,0.0180
7,Process_Velocity_1,6,0.08,0.1210,0.1770
8,Process_Pressure_Rise_Time,4,0.05,0.0255,0.0535
9,Process_Velocity_3,4,0.05,0.1495,0.2335


| outlier_ratio(%)     | 해석                    |
| ------ | --------------------- |
| 0~1%   | 거의 정상 범위              |
| 1~5%   | 제조 데이터에서 흔함           |
| 5~10%  | 공정 불안정 가능성            |
| 10% 이상 | 공정 세팅 문제 or 데이터 오류 의심 |


이상치 구간 vs 정상 구간 불량 비율 비교

In [14]:
import pandas as pd
import numpy as np

# Defect 관련 컬럼 제거
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = [col for col in numeric_cols 
                if not col.startswith('Defects') 
                and col != 'defect_any'
                and col != 'is_outlier']

results = []

for col in numeric_cols:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    is_outlier = ((df[col] < lower) | (df[col] > upper)).astype(int)
    
    grouped = df.groupby(is_outlier)['defect_any'].mean()
    
    if len(grouped) == 2:
        results.append({
            'column': col,
            'normal_defect_rate': round(grouped[0], 4),
            'outlier_defect_rate': round(grouped[1], 4),
            'difference(Outlier - Normal)': round(grouped[1] - grouped[0], 4)
        })

result_df = pd.DataFrame(results)
result_df = result_df.sort_values(by='difference(Outlier - Normal)', ascending=False)

result_df.head(10)

,column,normal_defect_rate,outlier_defect_rate,difference(Outlier - Normal)
9,Sensor_Factory_Temp,0.2218,0.2701,0.0483
5,Process_Rapid_Rise_Time,0.2242,0.2000,-0.0242
7,Process_Cycle_Time,0.2242,0.1818,-0.0424
6,Process_Biscuit_Thickness,0.2242,0.1667,-0.0576
1,Process_Velocity_2,0.2284,0.0543,-0.1741
0,Process_Velocity_1,0.2243,0.0000,-0.2243
2,Process_Velocity_3,0.2243,0.0000,-0.2243
4,Process_Cylinder_Pressure,0.2243,0.0000,-0.2243
8,Process_Pressure_Rise_Time,0.2243,0.0000,-0.2243
3,Process_High_Velocity,0.2245,0.0000,-0.2245


| difference | 해석                  |
| ---------- | ------------------- |
| 양수 (+)     | 이상치에서 불량 더 많이 발생  |
| 0 근처       | 영향 거의 없음            |
| 음수 (-)     | 오히려 정상 구간이 더 위험     |
